In [ ]:
import os
import time

from pprint import pprint
import pandas as pd
import numpy as np
from rtm_pymmcore.core.data_structures import SegmentationMethod
from rtm_pymmcore.core.conversion import load_events_json

## 1. Imports and paths

# Post-Experiment Reanalysis Pipeline

Re-process an already-acquired experiment **offline**, without a microscope
connection. This is useful when you want to:

- **Re-segment** images with a different or improved segmentator
  (set `use_old_segmentations=False` and pass new segmentators).
- **Re-track** cells with different tracker parameters.
- **Re-extract features** (e.g. switch from `SimpleFE` to `FE_ErkKtr`,
  or add a reference-image feature extractor).
- **Re-compute stimulation masks** with a different stimulator
  (set `use_old_stim_masks=False` and pass a new stimulator).
- **Reuse existing segmentations/stim masks** (`use_old_segmentations=True`,
  `use_old_stim_masks=True`) while only re-running tracking and feature
  extraction — useful when you want to change tracker settings or add
  new feature columns without the cost of re-segmentation.

The pipeline reads raw images (TIFF or OME-Zarr) from the original
experiment directory and writes all outputs (tracks, labels, particles,
stim masks) to a **new output directory**, leaving the original data
untouched.

## Key parameters

| Parameter | Effect |
|-----------|--------|
| `img_storage_path` | Path to the original experiment (with `raw/` or `acquisition.ome.zarr`) |
| `out_path` | Output directory for reanalysis results |
| `df_acquire` | DataFrame with one row per frame — must have `fov`, `timestep`, `fname` columns |
| `use_old_segmentations` | `True` → read existing label masks instead of re-segmenting |
| `use_old_stim_masks` | `True` → copy existing stim masks instead of re-computing |
| `n_jobs` | Number of parallel threads (one thread per FOV) |
| `correct_timestep_jumps` | `True` → backfill missing timesteps so tracking stays continuous |

In [ ]:
base_path = "E:\\Alex\\"
experiment_name = "DemoTestnewThread"
path = os.path.join(base_path, experiment_name)
events = load_events_json(path)
out_path = os.path.join(base_path, f"{experiment_name}_re")

## 2. Configure the processing pipeline

Choose which components to use for reanalysis. Any component can be
swapped out independently:

- **Segmentator**: `DummySegmentator` is a no-op (use with `use_old_segmentations=True`
  to reuse existing masks). Replace with e.g. `CellposeSegmentator` to re-segment.
- **Feature extractor**: `SimpleFE` extracts basic features (area, centroid,
  mean intensity). Replace with `FE_ErkKtr` for ERK/KTR ratio measurements.
- **Tracker**: `TrackerTrackpy` links cells across frames using trackpy.
- **Stimulator**: `StimWholeFOV` applies whole-FOV stimulation masks.
  Set to `None` if no stimulation was used.

In [ ]:
%load_ext autoreload
%autoreload 2

from rtm_pymmcore.segmentation.base import DummySegmentator
from rtm_pymmcore.stimulation.base import StimWholeFOV
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy
from rtm_pymmcore.feature_extraction.erk_ktr import FE_ErkKtr
from rtm_pymmcore.feature_extraction.simple import SimpleFE


segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=DummySegmentator(),
        use_channel=0,
        save_tracked=True,
    ),
]

stimulator = StimWholeFOV()
feature_extractor = SimpleFE("labels")
tracker = TrackerTrackpy()
ref_fe = None


from rtm_pymmcore.core.pipeline_post import ImageProcessingPipeline_postExperiment

pipeline = ImageProcessingPipeline_postExperiment(
    img_storage_path=path,
    out_path=out_path,
    events=events,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    feature_extractor_ref=ref_fe,
    use_old_segmentations=True,
    use_old_stim_masks=True,
    n_jobs=4,
)

In [4]:
pipeline.run()
pipeline.concat_fovs()

Processing FOV 0
Finished processing all FOVs.


## 3. Run the reanalysis

`pipeline.run()` processes each FOV in parallel (up to `n_jobs` threads).
For every frame it:

1. Reads the raw image (from `raw/` TIFFs or `acquisition.ome.zarr`).
2. Segments (or loads existing masks if `use_old_segmentations=True`).
3. Extracts cell positions and runs tracking.
4. Extracts features and merges them into the tracked DataFrame.
5. Optionally re-computes stimulation masks.
6. Saves labels, particles, and per-FOV tracking parquets to `out_path`.

`pipeline.concat_fovs()` merges all per-FOV parquets into a single
`exp_data.parquet` for downstream analysis.